# LMIP Metadata Seeding

**Purpose**: Seed initial metadata tables with source configurations, DQ rules, and taxonomy mappings

**Target Schema**: `workspace.metadata`

## Metadata Tables Initialized

| Table | Purpose |
|-------|----------|
| **source_config** | External API source configurations (Remotive, Arbeitnow) |
| **dq_rule_definitions** | Data quality rule definitions and thresholds |
| **taxonomy_role_canonical** | Canonical role taxonomy for job title mapping |
| **taxonomy_skill_catalog** | Master skill catalog with categorization |
| **pipeline_run_control** | Pipeline execution tracking and orchestration |

---

## Usage

**First-time initialization**:
```python
dbutils.notebook.run("/LMIP/notebooks/init/init_seed_metadata", timeout_seconds=600)
```

**Idempotent**: Uses MERGE to upsert seed data

In [0]:
# Databricks notebook source
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configuration
CATALOG = "workspace"
METADATA_SCHEMA = "metadata"

# Helper function for seeding
def seed_table(table_name, seed_data, merge_keys):
    """
    Seed metadata table with initial data using MERGE
    
    Args:
        table_name: Target table name (schema.table)
        seed_data: List of dictionaries with seed records
        merge_keys: List of columns to match on for MERGE
    """
    if not seed_data:
        print(f"⚠ No seed data provided for {table_name}")
        return
    
    # Create DataFrame from seed data
    seed_df = spark.createDataFrame(seed_data)
    
    # Create temp view for merge
    temp_view = f"temp_{table_name.split('.')[-1]}_seed"
    seed_df.createOrReplaceTempView(temp_view)
    
    # Build merge conditions
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in merge_keys])
    
    # Build update set clause
    columns = seed_df.columns
    update_set = ", ".join([f"{col} = source.{col}" for col in columns])
    
    # Build insert clause
    insert_columns = ", ".join(columns)
    insert_values = ", ".join([f"source.{col}" for col in columns])
    
    merge_sql = f"""
    MERGE INTO {table_name} AS target
    USING {temp_view} AS source
    ON {merge_condition}
    WHEN MATCHED THEN UPDATE SET {update_set}
    WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})
    """
    
    spark.sql(merge_sql)
    print(f"✓ Seeded {table_name} with {len(seed_data)} records")

print(f"Catalog: {CATALOG}")
print(f"Metadata Schema: {METADATA_SCHEMA}")
print(f"Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")

In [0]:
# Databricks notebook source
# Seed source_config table with initial API sources

source_config_data = [
    {
        "source_config_sk": 1,
        "source_name": "remotive",
        "source_type": "REST",
        "endpoint_url": "https://remotive.com/api/remote-jobs",
        "auth_ref": "none",  # Public API, no auth required
        "active_flag": True,
        "page_size": 100,
        "rate_limit_rpm": 60,
        "retry_attempts": 3,
        "timeout_seconds": 30,
        "created_at": datetime.now(timezone.utc),
        "updated_at": datetime.now(timezone.utc)
    },
    {
        "source_config_sk": 2,
        "source_name": "arbeitnow",
        "source_type": "REST",
        "endpoint_url": "https://www.arbeitnow.com/api/job-board-api",
        "auth_ref": "none",  # Public API, no auth required
        "active_flag": True,
        "page_size": 100,
        "rate_limit_rpm": 60,
        "retry_attempts": 3,
        "timeout_seconds": 30,
        "created_at": datetime.now(timezone.utc),
        "updated_at": datetime.now(timezone.utc)
    }
]

print("\nSeeding source_config...")
seed_table(
    table_name=f"{CATALOG}.{METADATA_SCHEMA}.source_config",
    seed_data=source_config_data,
    merge_keys=["source_name"]
)

In [0]:
# Databricks notebook source
# Seed dq_rule_definitions with initial data quality rules

dq_rule_data = [
    {
        "rule_id": "DQ001",
        "rule_name": "Job ID Not Null",
        "rule_type": "NOT_NULL",
        "target_schema": "bronze",
        "target_table": "bronze_job_snapshot",
        "target_column": "source_job_id",
        "rule_logic": "source_job_id IS NOT NULL",
        "severity": "CRITICAL",
        "threshold_value": 0.0,
        "active_flag": True,
        "created_at": datetime.now(timezone.utc)
    },
    {
        "rule_id": "DQ002",
        "rule_name": "Job Title Not Null",
        "rule_type": "NOT_NULL",
        "target_schema": "silver",
        "target_table": "silver_jobs_current",
        "target_column": "job_title",
        "rule_logic": "job_title IS NOT NULL AND TRIM(job_title) != ''",
        "severity": "CRITICAL",
        "threshold_value": 0.0,
        "active_flag": True,
        "created_at": datetime.now(timezone.utc)
    },
    {
        "rule_id": "DQ003",
        "rule_name": "Company Name Not Null",
        "rule_type": "NOT_NULL",
        "target_schema": "silver",
        "target_table": "silver_jobs_current",
        "target_column": "company_name",
        "rule_logic": "company_name IS NOT NULL AND TRIM(company_name) != ''",
        "severity": "HIGH",
        "threshold_value": 0.05,
        "active_flag": True,
        "created_at": datetime.now(timezone.utc)
    },
    {
        "rule_id": "DQ004",
        "rule_name": "Valid URL Format",
        "rule_type": "FORMAT",
        "target_schema": "silver",
        "target_table": "silver_jobs_current",
        "target_column": "job_url",
        "rule_logic": "job_url RLIKE '^https?://.*'",
        "severity": "MEDIUM",
        "threshold_value": 0.10,
        "active_flag": True,
        "created_at": datetime.now(timezone.utc)
    },
    {
        "rule_id": "DQ005",
        "rule_name": "Future Date Check",
        "rule_type": "RANGE",
        "target_schema": "silver",
        "target_table": "silver_jobs_current",
        "target_column": "posted_date",
        "rule_logic": "posted_date <= CURRENT_DATE()",
        "severity": "HIGH",
        "threshold_value": 0.02,
        "active_flag": True,
        "created_at": datetime.now(timezone.utc)
    }
]

print("\nSeeding dq_rule_definitions...")
seed_table(
    table_name=f"{CATALOG}.{METADATA_SCHEMA}.dq_rule_definitions",
    seed_data=dq_rule_data,
    merge_keys=["rule_id"]
)

In [0]:
# Databricks notebook source
# Seed taxonomy_role_canonical with initial canonical roles

role_taxonomy_data = [
    {"canonical_role_id": "ROLE_SE", "canonical_role_name": "Software Engineer", "role_category": "Engineering", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_SR_SE", "canonical_role_name": "Senior Software Engineer", "role_category": "Engineering", "seniority_level": "Senior"},
    {"canonical_role_id": "ROLE_JR_SE", "canonical_role_name": "Junior Software Engineer", "role_category": "Engineering", "seniority_level": "Junior"},
    {"canonical_role_id": "ROLE_DS", "canonical_role_name": "Data Scientist", "role_category": "Data & Analytics", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_SR_DS", "canonical_role_name": "Senior Data Scientist", "role_category": "Data & Analytics", "seniority_level": "Senior"},
    {"canonical_role_id": "ROLE_DA", "canonical_role_name": "Data Analyst", "role_category": "Data & Analytics", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_DE", "canonical_role_name": "Data Engineer", "role_category": "Data & Analytics", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_PM", "canonical_role_name": "Product Manager", "role_category": "Product", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_SR_PM", "canonical_role_name": "Senior Product Manager", "role_category": "Product", "seniority_level": "Senior"},
    {"canonical_role_id": "ROLE_FE", "canonical_role_name": "Frontend Engineer", "role_category": "Engineering", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_BE", "canonical_role_name": "Backend Engineer", "role_category": "Engineering", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_FS", "canonical_role_name": "Full Stack Engineer", "role_category": "Engineering", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_DEVOPS", "canonical_role_name": "DevOps Engineer", "role_category": "Engineering", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_MLE", "canonical_role_name": "Machine Learning Engineer", "role_category": "Data & Analytics", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_DES", "canonical_role_name": "UX/UI Designer", "role_category": "Design", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_QA", "canonical_role_name": "QA Engineer", "role_category": "Engineering", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_SEC", "canonical_role_name": "Security Engineer", "role_category": "Engineering", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_CSM", "canonical_role_name": "Customer Success Manager", "role_category": "Customer Success", "seniority_level": "Mid"},
    {"canonical_role_id": "ROLE_OTHER", "canonical_role_name": "Other", "role_category": "Uncategorized", "seniority_level": "Unknown"}
]

# Add timestamps
for role in role_taxonomy_data:
    role["created_at"] = datetime.now(timezone.utc)
    role["active_flag"] = True

print("\nSeeding taxonomy_role_canonical...")
seed_table(
    table_name=f"{CATALOG}.{METADATA_SCHEMA}.taxonomy_role_canonical",
    seed_data=role_taxonomy_data,
    merge_keys=["canonical_role_id"]
)

In [0]:
# Databricks notebook source
# Seed taxonomy_skill_catalog with initial skills

skill_catalog_data = [
    # Programming Languages
    {"skill_id": "SKILL_PYTHON", "skill_name": "Python", "skill_category": "Programming Language", "parent_skill_id": None},
    {"skill_id": "SKILL_JAVA", "skill_name": "Java", "skill_category": "Programming Language", "parent_skill_id": None},
    {"skill_id": "SKILL_JS", "skill_name": "JavaScript", "skill_category": "Programming Language", "parent_skill_id": None},
    {"skill_id": "SKILL_SQL", "skill_name": "SQL", "skill_category": "Programming Language", "parent_skill_id": None},
    {"skill_id": "SKILL_SCALA", "skill_name": "Scala", "skill_category": "Programming Language", "parent_skill_id": None},
    {"skill_id": "SKILL_R", "skill_name": "R", "skill_category": "Programming Language", "parent_skill_id": None},
    
    # Frameworks & Tools
    {"skill_id": "SKILL_SPARK", "skill_name": "Apache Spark", "skill_category": "Big Data", "parent_skill_id": None},
    {"skill_id": "SKILL_KAFKA", "skill_name": "Apache Kafka", "skill_category": "Big Data", "parent_skill_id": None},
    {"skill_id": "SKILL_AIRFLOW", "skill_name": "Apache Airflow", "skill_category": "Orchestration", "parent_skill_id": None},
    {"skill_id": "SKILL_DBT", "skill_name": "dbt", "skill_category": "Data Engineering", "parent_skill_id": None},
    {"skill_id": "SKILL_REACT", "skill_name": "React", "skill_category": "Frontend Framework", "parent_skill_id": "SKILL_JS"},
    {"skill_id": "SKILL_DJANGO", "skill_name": "Django", "skill_category": "Backend Framework", "parent_skill_id": "SKILL_PYTHON"},
    {"skill_id": "SKILL_FLASK", "skill_name": "Flask", "skill_category": "Backend Framework", "parent_skill_id": "SKILL_PYTHON"},
    
    # Cloud & Infrastructure
    {"skill_id": "SKILL_AWS", "skill_name": "Amazon Web Services", "skill_category": "Cloud Platform", "parent_skill_id": None},
    {"skill_id": "SKILL_AZURE", "skill_name": "Microsoft Azure", "skill_category": "Cloud Platform", "parent_skill_id": None},
    {"skill_id": "SKILL_GCP", "skill_name": "Google Cloud Platform", "skill_category": "Cloud Platform", "parent_skill_id": None},
    {"skill_id": "SKILL_DOCKER", "skill_name": "Docker", "skill_category": "Containerization", "parent_skill_id": None},
    {"skill_id": "SKILL_K8S", "skill_name": "Kubernetes", "skill_category": "Orchestration", "parent_skill_id": None},
    
    # Data & ML
    {"skill_id": "SKILL_PANDAS", "skill_name": "Pandas", "skill_category": "Data Manipulation", "parent_skill_id": "SKILL_PYTHON"},
    {"skill_id": "SKILL_NUMPY", "skill_name": "NumPy", "skill_category": "Data Manipulation", "parent_skill_id": "SKILL_PYTHON"},
    {"skill_id": "SKILL_SKLEARN", "skill_name": "Scikit-learn", "skill_category": "Machine Learning", "parent_skill_id": "SKILL_PYTHON"},
    {"skill_id": "SKILL_TENSORFLOW", "skill_name": "TensorFlow", "skill_category": "Machine Learning", "parent_skill_id": "SKILL_PYTHON"},
    {"skill_id": "SKILL_PYTORCH", "skill_name": "PyTorch", "skill_category": "Machine Learning", "parent_skill_id": "SKILL_PYTHON"},
]

# Add timestamps and active flag
for skill in skill_catalog_data:
    skill["created_at"] = datetime.now(timezone.utc)
    skill["active_flag"] = True

print("\nSeeding taxonomy_skill_catalog...")
seed_table(
    table_name=f"{CATALOG}.{METADATA_SCHEMA}.taxonomy_skill_catalog",
    seed_data=skill_catalog_data,
    merge_keys=["skill_id"]
)

In [0]:
# Databricks notebook source
# Verify seeding results

print("\n" + "="*60)
print("METADATA SEEDING VERIFICATION")
print("="*60)

tables_to_verify = [
    "source_config",
    "dq_rule_definitions",
    "taxonomy_role_canonical",
    "taxonomy_skill_catalog"
]

verification_results = []

for table in tables_to_verify:
    full_table_name = f"{CATALOG}.{METADATA_SCHEMA}.{table}"
    try:
        count = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_table_name}").collect()[0].cnt
        verification_results.append((table, count, "SUCCESS"))
        print(f"✓ {table}: {count} records")
    except Exception as e:
        verification_results.append((table, 0, "FAILED"))
        print(f"✗ {table}: FAILED - {str(e)}")

print("="*60)

# Check if all succeeded
all_success = all([result[2] == "SUCCESS" for result in verification_results])

if all_success:
    status = "SUCCESS"
    print("\n✓ All metadata tables seeded successfully")
else:
    status = "PARTIAL"
    print("\n✗ Some metadata tables failed to seed")

print(f"\nCompleted: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")

# Return status for orchestration
dbutils.notebook.exit(status)